# Tutorial

The `pvanalysis` package of `SLAM` is a tool to identify Keplerian disks in protostellar systems using position-velocity (PV) diagrams and estimate the dynamical mass of protostars if disks are present. This tool basically consists of two steps: `get_edgeridge`, which determines edge/ridge points that trace rotation curve features of PV diagrams, and `fit_edgeridge`, which performs the power-law fitting with the obtained edge/ridge points. In this note, we will briefly present how to use this tool. Users can refer to the [manual](https://ui.adsabs.harvard.edu/abs/2023arXiv231107988A/abstract) for full details.

In [ ]:
import numpy as np
# if you need to set a path
import sys
sys.path.append('../../../SLAM/')  # add PATH to SLAM

from pvanalysis import PVAnalysis

## Basic usage

Here is an example only with the most basic input parameters to demonstrate the simplest usage. The first step of `pvanalysis` is to extract edge/ridge points from a PV diagram.

In [ ]:
# -------- INPUTS --------
fitsfile = '../../testfits/test.pvanalysis.fits'
outname = 'pvanalysis'  # file name header for outputs
incl = 48.  # inclination angle (deg)
vsys = 6.4  # systemic velocity (km/s)
dist = 140.  # distance to the object (pc)
rms = 1.7e-3  # rms noise level (Jy/beam)
cutoff = 5.  # threshold for noise cut-off for edge/ridge calculations (rms)
# -------------------------


# read PV diagram
# give rms, vsys, distance, and inclination angle
impv = PVAnalysis(fitsfile, rms, vsys, dist, incl=incl, pa=None)
# get edge/ridge points
impv.get_edgeridge(outname, cutoff=cutoff,)
impv.write_edgeridge(outname=outname)

In this example, edge/ridge points are obtained from emission above the given threshold ($5\sigma$) in the entire position-velocity (PV) diagram.

In the second step below, the power-law fitting is done with the obtained edge/ridge points. 

In [ ]:
# power law fitting
# --------- input parameters ----------
include_vsys = False  # False fixes the offset from input vsys at zero.
include_dp = True  # False means a single power
include_pin = False  # False means pin=0.5 (Keplerian).
fixed_pin = 0.5  # Set the fixed pin value when include_pin is False.
fixed_dp = 0.0  # Set the fixed dp value when include_dp is False.
show_corner = True  # if show corner plots or not
# -------------------------------------
impv.fit_edgeridge(include_vsys=include_vsys,
                   include_dp=include_dp,
                   include_pin=include_pin,
                   fixed_pin=fixed_pin, fixed_dp=fixed_dp,
                   outname=outname, rangelevel=0.8,
                   show_corner=show_corner, save_corner=True,
                   kwargs_emcee_corner={'nburnin': 2000, 'nsteps': 2000})
impv.output_fitresult()

The input parameters set free parameters and which model function (single or double power-law) is adopted. In the above case, the fitting model is a double-power law function with a fixed inner power-law index ($p_\mathrm{in}=0.5$). The fitting searches the best break point ($R_\mathrm{b}$, $V_\mathrm{b}$), where the power-law index changes, and $dp$, which is deviation of the outer power-law index from the innder one. The dynamical mass ($M_\mathrm{b}$) is estimated from the set of ($R_\mathrm{b}$, $V_\mathrm{b}$) and a given inclination angle assuming a Keplerian rotation.

The edge/ridge points and the best-fit functions can be visualized as follows.

In [ ]:
# plot results
impv.plot_fitresult(outname=outname, clevels=[-9, -6, -3, 3, 6, 9],
                    kwargs_pcolormesh={'cmap': 'viridis'},
                    kwargs_contour={'colors': 'lime'},
                    fmt={'edge': 'v', 'ridge': 'o'},
                    linestyle={'edge': '--', 'ridge': '-'},)

## Advanced usage


### Detailed Setup
Here is a more advanced example with a full set of main parameters.

In [ ]:
# -------- INPUTS --------
# Parameters of the target source and PV diagram
fitsfile = '../../testfits/test.pvanalysis.fits'
outname = 'pvanalysis'  # file name header for outputs
incl = 48.  # inclination angle (deg)
pa = None  # position angle (deg), used to calculate angular resolution along the cut direction if provided
vsys = 6.4  # systemic velocity (km/s)
dist = 140.  # distance (pc)
rms = 1.7e-3  # rms noise level (Jy/beam)
cutoff = 5.  # noise cut-off threshold for edge/ridge calculations (rms)

# Parameters for calculations of edge/ridge points
ridgemode = 'mean'  # 'mean' or 'gauss'
xlim = [-200, 0, 0, 200]  # au; [-outlim, -inlim, inlim, outlim]
vlim = np.array([-5, 0, 0, 5]) + vsys  # km/s
Mlim = [0, 10]  # M_sun; to exclude unreasonable points
use_velocity = True  # cuts along the velocity direction
use_position = True  # cuts along the positional direction
minabserr = 0  # minimum absolute errorbar in the unit of bmaj or dv.
minrelerr = 0  # minimum relative errorbar.
nanbeforemax = True
nanopposite = True
nanbeforecross = True

# Parameters for power-law fitting
include_vsys = False  # False fixes the offset from input vsys at zero.
include_dp = True  # False means a single power
include_pin = False  # False means pin=0.5 (Keplerian).
fixed_pin = 0.5  # Fixed pin when include_pin is False.
fixed_dp = 0.0  # Fixed dp when include_dp is False.
show_corner = True  # figures will be made regardless of this option.
calc_evidence = False  # Optional Bayesian evidence calculation.

# for plot
xlim_plot = [200. / 20., 200.]  # au; [inlim, outlim]
vlim_plot = [6. / 20., 6.]  # km/s
# ------------------------

These input parameters are to tune how to select edge/ridge points that better represent rotation curves, and free and fixed parameters for the power-law fitting.

With these input parameters, the analysis goes as follows.

In [ ]:
# run all once
impv = PVAnalysis(fitsfile, rms, vsys, dist, incl=incl, pa=None)
impv.get_edgeridge(outname, cutoff=cutoff, ridgemode=ridgemode,
                   use_position=use_position, use_velocity=use_velocity,
                   Mlim=Mlim, xlim=np.array(xlim) / dist, vlim=vlim,
                   minabserr=minabserr, minrelerr=minrelerr,
                   nanbeforemax=nanbeforemax, nanopposite=nanopposite, 
                   nanbeforecross=nanbeforecross)
impv.write_edgeridge(outname=outname)
impv.fit_edgeridge(include_vsys=include_vsys,
                   include_dp=include_dp,
                   include_pin=include_pin,
                   fixed_pin=fixed_pin, fixed_dp=fixed_dp,
                   outname=outname, rangelevel=0.8,
                   show_corner=show_corner, calc_evidence=calc_evidence)
impv.output_fitresult()
impv.plot_fitresult(vlim=vlim_plot, xlim=xlim_plot, flipaxis=False,
                    clevels=[-9, -6, -3, 3, 6, 9], outname=outname,
                    show=True, logcolor=True, Tbcolor=False, vsys_mode='input',
                    kwargs_pcolormesh={'cmap': 'viridis'},
                    kwargs_contour={'colors': 'lime'},
                    fmt={'edge': 'v', 'ridge': 'o'},
                    linestyle={'edge': '--', 'ridge': '-'},
                    plotridgepoint=True, plotedgepoint=True,
                    plotridgemodel=True, plotedgemodel=True)

### Prior and Posterior

The uniform distribution is adopted for the prior in the MCMC fitting. The preset prior for $R_\mathrm{b}$ and $V_\mathrm{b}$ is over the radial and velocity ranges of the obtained ridge and edge points, respectively. The preset prior for $p_\mathrm{in}$, $dp$ and $V_\mathrm{sys}$ are $0.01 < p_\mathrm{in} < 10$, $0 < dp < 10$ and $V_\mathrm{sys,given} - 1 < V_\mathrm{sys} < V_\mathrm{sys,given} + 1$, respectively. These parameter ranges sufficiently cover typical values of the parameters. The top 80\% of the posterior will be plotted with the default setting.

One might want to modify the prior and plots of the posterior in some cases, e.g., to check and omit local minima other than the best solution. The prior and plot setting for the posterior can be manually set as follows:

In [ ]:
impv.fit_edgeridge(include_vsys=include_vsys,
                   include_dp=include_dp,
                   include_pin=include_pin,
                   fixed_pin=fixed_pin, fixed_dp=fixed_dp,
                   outname=outname,
                   show_corner=show_corner,
                   # parameters to manually set the prior and change plot setting for the posterior
                   rb_range=[90., 140.],  # prior range for Rb
                   vb_range=[1., 5.],  # for Vb
                   pin_range=[0.1, 10.],  # for p_in
                   dp_range=[0., 2.],  # for dp
                   vsys_range=[-0.5, 0.5],  # for Vsys
                   rangelevel=1.0,  # top rangelevel x 100 % of the posterior will be plotted
                  )
impv.output_fitresult()

## Reuse saved points and fit independent methods

Extraction and fitting are separate. Save points once, then load them into a
new object without re-extracting or applying extraction filters again. Both
methods are selected by default; `methods=('ridge',)` needs only the ridge file.
The plotter works before fitting and automatically skips unavailable overlays.


In [ ]:
# Use the points extracted above, independently of previous fitted models.
impv.write_edgeridge(outname='saved_points')
refit = PVAnalysis(fitsfile, rms, vsys, dist, incl=incl)
refit.read_edgeridge(outname='saved_points')
refit.plot_fitresult(outname='saved_points_preview', show=False,
                     xlim=xlim_plot, vlim=vlim_plot, vsys_mode='input')


### Repeat fitting without repeating extraction or loading

These calls try a single power law with a free index and a broken power law with
a fixed inner index, using the same ridge points. Distinct output names retain
each trial's files. Sampling controls and corner-saving settings apply to every
method selected in a call; separate edge/ridge calls allow different settings.
An empty or unavailable method has a `None` result, so check it before indexing.
The sampling lengths here illustrate configuration, rather than establish
convergence for a scientific result.


In [ ]:
sampling = {'nburnin': 500, 'nsteps': 1000}
single = refit.fit_edgeridge(
    methods=('ridge',), include_dp=False, include_pin=True,
    outname='ridge_single', save_corner=False, kwargs_emcee_corner=sampling,
)
broken = refit.fit_edgeridge(
    methods=('ridge',), include_dp=True, include_pin=False,
    outname='ridge_broken', save_corner=False, kwargs_emcee_corner=sampling,
)
if broken['ridge'] is not None:
    print('Ridge parameters:', broken['ridge']['popt'])


In [ ]:
# This replaces only the edge fit; the ridge_broken fit remains available.
edge = refit.fit_edgeridge(
    methods=('edge',), include_dp=False,
    outname='edge_single', save_corner=True,
    kwargs_emcee_corner={'nburnin': 500, 'nsteps': 1500},
)
refit.output_fitresult()
refit.plot_fitresult(outname='independent_models', show=False,
                     xlim=xlim_plot, vlim=vlim_plot, vsys_mode='input')


### Plot an image before extraction

No dummy points or fit parameters are required. The default plot center is the
input `vsys`. After successful power-law fits, `vsys_mode='edge'`, `'ridge'`, or
`'mean'` selects fitted offsets. Explicit edge/ridge modes require that fit;
mean uses available offsets and falls back to input vsys if none exist.


In [ ]:
preview = PVAnalysis(fitsfile, rms, vsys, dist, incl=incl)
preview.plot_fitresult(outname='image_only', show=False,
                       xlim=xlim_plot, vlim=vlim_plot)
